# WhatsApp to Excel - FIXED Manual Login Version

یہ **fixed version** ہے جو **manual login** کے ساتھ کام کرتا ہے۔


In [ ]:
# Step 1: Install packages
import subprocess
import sys

packages = ['selenium', 'openpyxl', 'webdriver-manager', 'pandas']

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"OK {package} installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print("\nOK All packages ready!")

In [ ]:
# Step 2: Import libraries
import os
import time
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import openpyxl

print("OK Libraries imported!")

In [ ]:
# Step 3: Define class
class WhatsAppCollector:
    def __init__(self, excel_file_path: str, start_date: str, end_date: str):
        self.excel_file_path = excel_file_path
        self.start_date = start_date
        self.end_date = end_date
        self.driver = None
        self.processed = set()
        
    def parse_date(self, date_str: str):
        try:
            parts = date_str.split('-')
            return datetime(int(parts[2]), int(parts[1]), int(parts[0]))
        except:
            return None
    
    def is_in_range(self, msg_date_str: str) -> bool:
        msg_date = self.parse_date(msg_date_str)
        start = self.parse_date(self.start_date)
        end = self.parse_date(self.end_date)
        
        if not all([msg_date, start, end]):
            return False
        
        return start <= msg_date <= end
    
    def setup_browser(self):
        print("\nOpening WhatsApp Web...\n")
        
        options = Options()
        options.add_argument("--start-maximized")
        
        user_dir = str(Path.home() / ".whatsapp_collector")
        options.add_argument(f"user-data-dir={user_dir}")
        
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=options)
        
        self.driver.get("https://web.whatsapp.com")
        
        print("!" * 60)
        print("MANUAL LOGIN REQUIRED:")
        print("1. Scan QR code with your phone")
        print("2. Open your complaint group")
        print("3. Scroll up to see old messages")
        print("4. Press Enter below")
        print("!" * 60 + "\n")
        
        input("Press Enter when ready...")
        print("\nOK Starting collection...\n")
        time.sleep(2)
    
    def scroll_and_collect(self, scroll_count: int = 15):
        print(f"Scrolling and collecting messages...\n")
        
        for scroll_num in range(scroll_count):
            print(f"Scroll #{scroll_num + 1}/{scroll_count}...", end=' ')
            
            try:
                # Scroll up
                body = self.driver.find_element(By.TAG_NAME, "body")
                body.send_keys(Keys.HOME)
                time.sleep(1)
                
                # Extract messages
                messages = self.driver.find_elements(By.XPATH, "//div[@data-testid='msg-container']")
                
                new_count = 0
                for msg in messages:
                    try:
                        text = msg.text
                        if text and len(text) > 10:
                            msg_id = hash(text) % ((2**31) - 1)
                            if msg_id not in self.processed:
                                self.processed.add(msg_id)
                                complaint = self.parse_message(text)
                                if complaint:
                                    self.add_to_excel(complaint)
                                    new_count += 1
                    except:
                        pass
                
                print(f"Found {new_count} new")
                time.sleep(1)
                
            except Exception as e:
                print(f"Error: {e}")
    
    def parse_message(self, text: str) -> Optional[Dict]:
        lines = text.strip().split('\n')
        
        if len(lines) < 4:
            return None
        
        data = {}
        
        # Time
        time_match = re.search(r'(\d{1,2}:\d{2}\s*(?:am|pm|AM|PM)?)', lines[0])
        if time_match:
            data['Time'] = time_match.group(0)
        else:
            return None
        
        # Date
        date_match = re.search(r'(\d{1,2})-(\d{1,2})-(\d{4})', text)
        if date_match:
            data['Date'] = date_match.group(0)
        else:
            return None
        
        # Check date range
        if not self.is_in_range(data['Date']):
            return None
        
        # Branch
        branch_match = re.search(r'([A-Za-z\s]+?)\s+Branch', text, re.IGNORECASE)
        if branch_match:
            data['Branch'] = branch_match.group(1).strip()
        else:
            data['Branch'] = 'Unknown'
        
        # Complaint
        complaint = '\n'.join(lines[3:]).strip()
        if complaint:
            data['Complain'] = complaint[:200]
        else:
            return None
        
        # Category
        complaint_lower = complaint.lower()
        if 'staff' in complaint_lower or 'late' in complaint_lower:
            data['Category'] = 'staff issue'
        elif 'food' in complaint_lower or 'quality' in complaint_lower:
            data['Category'] = 'quality issue'
        elif 'service' in complaint_lower or 'slow' in complaint_lower:
            data['Category'] = 'service issue'
        else:
            data['Category'] = 'other'
        
        data['Response (Y / N)'] = 'N'
        
        return data
    
    def add_to_excel(self, data: Dict):
        try:
            wb = openpyxl.load_workbook(self.excel_file_path)
            ws = wb.active
            
            row = ws.max_row + 1
            
            cols = {'Date': 1, 'Time': 2, 'Branch': 3, 'Category': 4, 'Complain': 7, 'Response (Y / N)': 8}
            
            for key, col in cols.items():
                if key in data:
                    ws.cell(row=row, column=col, value=data[key])
            
            wb.save(self.excel_file_path)
            print(f"    ✓ Added: {data.get('Complain', '')[:40]}...")
            
        except Exception as e:
            print(f"    ERROR: {e}")
    
    def run(self):
        try:
            self.setup_browser()
            self.scroll_and_collect(scroll_count=20)
        finally:
            if self.driver:
                self.driver.quit()
            print("\nOK Done!")

print("OK Class defined!")

In [ ]:
# Step 4: Configuration

EXCEL_FILE = r"C:\Users\GT-Tech\Desktop\Feb_surveillance.xlsm.xlsx"

# Change these dates for different days
START_DATE = "1-4-2026"
END_DATE = "1-4-2026"

print(f"Excel: {EXCEL_FILE}")
print(f"Date Range: {START_DATE} to {END_DATE}")
print("\nOK Ready!")

In [ ]:
# Step 5: Run collection

if not os.path.exists(EXCEL_FILE):
    print(f"ERROR File not found: {EXCEL_FILE}")
else:
    print(f"OK File found\n")
    
    collector = WhatsAppCollector(EXCEL_FILE, START_DATE, END_DATE)
    collector.run()

## اگلے دن کے لیے:

Cell 4 میں تاریخ بدلیں:

```python
START_DATE = "2-4-2026"
END_DATE = "2-4-2026"
```

پھر Cell 5 دوبارہ چلائیں۔ نیا data Excel میں شامل ہو جائے گا! ✅